# Tutorial 9-3: High-Performance Nearest Neighbor Libraries
**Course:** CSEN 140: Machine Learning/Data Mining  
**Instructor:** Dr. David C. Anastasiu

---
## 1. Introduction to Vector Search Engines
In the previous tutorials, we looked at the mathematical pruning (L2-norm) and hashing (LSH) that make search faster. In industry, these concepts are implemented in highly optimized libraries. 

**Objective:** We will compare three major libraries on the same dataset:
1. **Scikit-learn (BallTree):** A standard exact-search baseline.
2. **Annoy (Spotify):** A tree-based approximate search library.
3. **FAISS (Meta):** A state-of-the-art library for approximate search using Quantization and Inverted Files.

We will measure **Indexing Time**, **Search Time**, and **Recall** (accuracy relative to brute force).

In [ ]:
# Install necessary packages
!pip install faiss-cpu annoy scikit-learn matplotlib

import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.neighbors import BallTree
import annoy
import faiss

# 1. Load Data
print("Loading dataset...")
newsgroups = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
data = newsgroups.data[:10000] # Use 10k documents

# 2. Vectorize and make Dense
# Most optimized libraries like FAISS and Annoy prefer dense vectors.
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X_sparse = vectorizer.fit_transform(data)

# Reduce dimensionality to 128 dense features for better library performance
pca = PCA(n_components=128)
X = pca.fit_transform(X_sparse.toarray()).astype('float32')

# Define queries (first 100 docs)
queries = X[:100]
k = 10 # Search for top 10 neighbors

print(f"Data prepared: {X.shape[0]} items, {X.shape[1]} features.")

## 2. Benchmarking Exact Search (Baseline)
We use Scikit-learn's BallTree to establish our 'Ground Truth' and baseline timing.

In [ ]:
print("Running Scikit-learn BallTree (Exact)...")
start = time.time()
tree = BallTree(X)
idx_time_sk = time.time() - start

start = time.time()
dist, ground_truth = tree.query(queries, k=k)
query_time_sk = time.time() - start

print(f"Indexing: {idx_time_sk:.4f}s | Query: {query_time_sk:.4f}s")

## 3. Benchmarking Annoy (Spotify)
Annoy builds a forest of trees. More trees increase accuracy (Recall) but also increase index size and time.

In [ ]:
print("Running Annoy (Approximate)...")
f = X.shape[1]
t = annoy.AnnoyIndex(f, 'euclidean')

start = time.time()
for i in range(len(X)):
    t.add_item(i, X[i])
t.build(10) # 10 trees
idx_time_annoy = time.time() - start

start = time.time()
annoy_results = []
for q in queries:
    annoy_results.append(t.get_nns_by_vector(q, k))
query_time_annoy = time.time() - start

# Calculate Recall
matches = 0
for i in range(len(queries)):
    matches += len(set(ground_truth[i]) & set(annoy_results[i]))
recall_annoy = matches / (len(queries) * k)

print(f"Indexing: {idx_time_annoy:.4f}s | Query: {query_time_annoy:.4f}s | Recall: {recall_annoy:.2%}")

## 4. Benchmarking FAISS (Meta)
FAISS uses an Inverted File (IVF) index with Voronoi cells. We partition the space into 100 cells (`nlist`) and search the top 10 most relevant cells (`nprobe`).

In [ ]:
print("Running FAISS IVF (Approximate)...")
d = X.shape[1]
nlist = 100
quantizer = faiss.IndexFlatL2(d)
index = faiss.IndexIVFFlat(quantizer, d, nlist)

start = time.time()
index.train(X) # IVF requires training to define Voronoi cells
index.add(X)
idx_time_faiss = time.time() - start

start = time.time()
index.nprobe = 10 # How many nearby cells to search
D, faiss_results = index.search(queries, k)
query_time_faiss = time.time() - start

# Calculate Recall
matches = 0
for i in range(len(queries)):
    matches += len(set(ground_truth[i]) & set(faiss_results[i]))
recall_faiss = matches / (len(queries) * k)

print(f"Indexing: {idx_time_faiss:.4f}s | Query: {query_time_faiss:.4f}s | Recall: {recall_faiss:.2%}")

## 5. Performance Comparison
We now visualize the trade-offs. Note that while indexing time varies, the query speed advantage of FAISS and Annoy becomes exponential as the number of documents grows.

In [ ]:
labels = ['Sklearn', 'Annoy', 'FAISS']
q_times = [query_time_sk, query_time_annoy, query_time_faiss]
recalls = [1.0, recall_annoy, recall_faiss]

fig, ax1 = plt.subplots(figsize=(10, 6))

ax1.set_xlabel('Library')
ax1.set_ylabel('Query Time (s)', color='tab:blue')
ax1.bar(labels, q_times, color='tab:blue', alpha=0.6, label='Query Time')
ax1.tick_params(axis='y', labelcolor='tab:blue')
ax1.set_yscale('log')

ax2 = ax1.twinx()
ax2.set_ylabel('Recall (%)', color='tab:red')
ax2.plot(labels, recalls, color='tab:red', marker='o', linewidth=2, label='Recall')
ax2.tick_params(axis='y', labelcolor='tab:red')
ax2.set_ylim(0, 1.1)

plt.title('Performance vs. Accuracy Trade-off')
plt.show()

## Summary
* **Exact Search (Sklearn):** Guaranteed correctness but slowest query time.
* **Annoy:** Excellent for scenarios where you need to save an index to a file and share it among multiple web servers.
* **FAISS:** The fastest and most customizable. It allows for advanced techniques like Product Quantization to compress vectors, though at a higher recall penalty.
* **The Trade-off:** In real-world systems (like recommendation engines), a 95% recall is usually acceptable if it means queries are 100x faster.